# Batch HYSPLIT untuk Pengisian Target Dataset

Notebook ini versi bersih untuk memproses dataset gabungan gunung berapi secara looping batch.

Output yang diisi otomatis per baris dataset:
- `jarak_km`
- `luas_km2`
- `sudut_deg`
- `radius_km`

Catatan:
- Runtime simulasi dipatok statis **1 jam** untuk semua baris.
- Lokasi erupsi per baris diambil dari kolom `latitude` dan `longitude`.
- Notebook ini tidak bergantung pada notebook lama.

In [ ]:
import glob
import os
import subprocess
from datetime import datetime
from math import radians, cos, sin, asin, sqrt, atan2, degrees
from pathlib import Path

import numpy as np
import pandas as pd

# =========================================================
# KONFIGURASI UTAMA
# =========================================================
DATASET_PATH = Path("d:/Projects/volcanic_ash/simulation/dataset-java-ash.csv")
OUTPUT_PATH = Path("d:/Projects/volcanic_ash/simulation/dataset-java-ash_hysplit_filled.csv")

HYSPLIT_EXE = "F:/HYSPLIT/exec/hycs_std.exe"
CON2ASC_EXE = "F:/HYSPLIT/exec/con2asc.exe"
WORKING_DIR = "F:/HYSPLIT/working"
METDATA_DIR = "F:/HYSPLIT/metdata"

# Runtime simulasi statis 1 jam
BATCH_RUNTIME_JAM = 2

# Parameter partikel/model default
EMISSION_RATE = 1000.0
PARTICLE_DIAMETER_UM = 30.0
PARTICLE_DENSITY_G_CM3 = 2.5
PARTICLE_SHAPE = 0.7
DRY_DEPOSITION_VEL = 0.01
WET_REMOVAL_COEF = 0.0
THRESHOLD_CONC = 1e-10
GRID_SPACING = 0.05
GRID_SPAN_LAT = 30.0
GRID_SPAN_LON = 30.0
TOP_OF_MODEL_M = 3000.0
VERTICAL_MOTION = 0

# Batch options
CHECKPOINT_EVERY = 50
MAX_ROWS = None  # set angka, misal 3, untuk uji cepat
OVERWRITE_OUTPUT = True

TARGET_COLS = ["jarak_km", "luas_km2", "sudut_deg", "radius_km"]
REQUIRED_INPUT_COLS = ["timestamp", "latitude", "longitude", "tinggi_letusan_m"]

In [2]:
def parse_row_timestamp(value):
    if pd.isna(value):
        raise ValueError("timestamp kosong")

    text = str(value).strip()
    formats = [
        "%Y-%m-%d %H:%M:%S UTC",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue

    dt = pd.to_datetime(text, utc=True, errors="coerce")
    if pd.isna(dt):
        raise ValueError(f"format timestamp tidak dikenali: {value}")
    return dt.to_pydatetime().replace(tzinfo=None)


def to_hysplit_timestamp(dt):
    return dt.strftime("%y %m %d %H")


def infer_gdas_met_file(dt):
    month = dt.strftime("%b").lower()
    # Modifikasi: Selalu gunakan tahun 25 (2025) dan minggu ke-1 (w1)
    yy = "25"
    week = 1
    return f"gdas1.{month}{yy}.w{week}"


def create_control_file(lat_sumber, lon_sumber, tinggi_letusan_m, timestamp, met_file):
    os.makedirs(WORKING_DIR, exist_ok=True)
    control_path = os.path.join(WORKING_DIR, "CONTROL")
    conc_level = max(float(tinggi_letusan_m), 1000.0)

    control_lines = [
        timestamp,
        "1",
        f"{lat_sumber} {lon_sumber} {tinggi_letusan_m}",
        str(BATCH_RUNTIME_JAM),
        str(VERTICAL_MOTION),
        f"{TOP_OF_MODEL_M}",
        "1",
        f"{METDATA_DIR}/",
        met_file,
        "1",
        "ASH",
        str(EMISSION_RATE),
        f"{BATCH_RUNTIME_JAM}.0",
        "00 00 00 00 00",
        "1",
        f"{lat_sumber} {lon_sumber}",
        f"{GRID_SPACING} {GRID_SPACING}",
        f"{GRID_SPAN_LAT} {GRID_SPAN_LON}",
        "./",
        "cdump",
        "1",
        f"{conc_level:.1f}",
        "00 00 00 00 00",
        "00 00 00 00 00",
        "00 01 00",
        "1",
        f"{PARTICLE_DIAMETER_UM} {PARTICLE_DENSITY_G_CM3} {PARTICLE_SHAPE}",
        f"{DRY_DEPOSITION_VEL} 0.0 0.0 0.0 0.0",
        f"{WET_REMOVAL_COEF} 0.0 0.0",
        "0.0",
        "0.0",
    ]

    with open(control_path, "w") as f:
        f.write("\n".join(control_lines) + "\n")


def run_hysplit_once(met_file):
    met_path = os.path.join(METDATA_DIR, met_file)
    if not os.path.exists(HYSPLIT_EXE):
        raise FileNotFoundError(f"Executable HYSPLIT tidak ditemukan: {HYSPLIT_EXE}")
    if not os.path.exists(met_path):
        raise FileNotFoundError(f"File meteo tidak ditemukan: {met_path}")

    result = subprocess.run(
        [HYSPLIT_EXE],
        cwd=WORKING_DIR,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        msg = result.stderr.strip() if result.stderr else "simulasi gagal"
        raise RuntimeError(msg)


def convert_cdump_to_txt():
    cdump_file = os.path.join(WORKING_DIR, "cdump")
    if not os.path.exists(cdump_file):
        raise FileNotFoundError(f"cdump tidak ditemukan: {cdump_file}")
    if not os.path.exists(CON2ASC_EXE):
        raise FileNotFoundError(f"con2asc.exe tidak ditemukan: {CON2ASC_EXE}")

    old_files = glob.glob(os.path.join(WORKING_DIR, "cdump_*"))
    for old_file in old_files:
        if os.path.isfile(old_file):
            os.remove(old_file)

    result = subprocess.run(
        [CON2ASC_EXE, "-i" + cdump_file],
        cwd=WORKING_DIR,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        msg = result.stderr.strip() if result.stderr else "konversi cdump gagal"
        raise RuntimeError(msg)

    converted_files = glob.glob(os.path.join(WORKING_DIR, "cdump_*"))
    converted_files = [f for f in converted_files if os.path.isfile(f) and not f.endswith(".txt")]
    if not converted_files:
        raise RuntimeError("file hasil konversi cdump_* tidak ditemukan")

    cdump_txt = os.path.join(WORKING_DIR, "cdump.txt")
    all_lines = []
    for conv_file in sorted(converted_files):
        with open(conv_file, "r") as f:
            all_lines.extend(f.readlines())

    with open(cdump_txt, "w") as f:
        f.writelines(all_lines)

    return cdump_txt

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    return R * c


def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = sin(dlon) * cos(lat2)
    y = cos(lat1) * sin(lat2) - sin(lat1) * cos(lat2) * cos(dlon)
    bearing = atan2(x, y)
    return (degrees(bearing) + 360) % 360


def parse_cdump_with_timestep(filepath):
    data_points = []
    with open(filepath, "r") as f:
        for line in f:
            stripped = line.strip()
            if not stripped or "DAY" in stripped:
                continue

            parts = stripped.split()
            numeric_values = []
            for part in parts:
                try:
                    numeric_values.append(float(part))
                except ValueError:
                    continue

            if len(numeric_values) >= 5:
                data_points.append({
                    "day": int(numeric_values[0]),
                    "hr": int(numeric_values[1]),
                    "lat": numeric_values[2],
                    "lon": numeric_values[3],
                    "concentration": numeric_values[4],
                })
    return data_points


def calculate_grid_area_km2(lats, lons):
    unique_lats = sorted(set(lats))
    unique_lons = sorted(set(lons))

    dlat_deg = abs(unique_lats[1] - unique_lats[0]) if len(unique_lats) >= 2 else 0.25
    dlon_deg = abs(unique_lons[1] - unique_lons[0]) if len(unique_lons) >= 2 else 0.25

    mean_lat = np.mean(unique_lats)
    dlat_km = dlat_deg * 111.32
    dlon_km = dlon_deg * 111.32 * cos(radians(abs(mean_lat)))
    return dlat_km * dlon_km


def analyze_cdump(cdump_txt, source_lat, source_lon, threshold_conc=THRESHOLD_CONC):
    raw_points = parse_cdump_with_timestep(cdump_txt)
    if len(raw_points) == 0:
        raise RuntimeError("cdump.txt tidak memiliki data")

    valid_points = []
    for p in raw_points:
        lat, lon, conc = p["lat"], p["lon"], p["concentration"]
        if -90 <= lat <= 90 and -180 <= lon <= 180 and conc > threshold_conc:
            valid_points.append(p)

    if len(valid_points) == 0:
        raise RuntimeError("tidak ada data konsentrasi di atas threshold")

    df_all = pd.DataFrame(valid_points)
    df_unique = df_all.groupby(["lat", "lon"]).agg(
        max_conc=("concentration", "max"),
        mean_conc=("concentration", "mean"),
        n_timesteps=("concentration", "count"),
    ).reset_index()

    distances = []
    bearings = []
    max_distance = 0.0

    for _, r in df_unique.iterrows():
        dist = haversine(source_lat, source_lon, r["lat"], r["lon"])
        bear = calculate_bearing(source_lat, source_lon, r["lat"], r["lon"])
        distances.append(dist)
        bearings.append(bear)
        if dist > max_distance:
            max_distance = dist

    df_unique["distance_km"] = distances

    jarak_km = float(max_distance)
    radius_km = float(df_unique["distance_km"].mean())

    rad_bearings = [radians(b) for b in bearings]
    mean_sin = np.mean([sin(r) for r in rad_bearings])
    mean_cos = np.mean([cos(r) for r in rad_bearings])
    sudut_deg = float(degrees(atan2(mean_sin, mean_cos)) % 360)

    grid_area_km2 = calculate_grid_area_km2(df_unique["lat"].values, df_unique["lon"].values)
    luas_km2 = float(len(df_unique) * grid_area_km2)

    return {
        "jarak_km": jarak_km,
        "luas_km2": luas_km2,
        "sudut_deg": sudut_deg,
        "radius_km": radius_km,
    }

In [4]:
def run_one_row(row):
    dt = parse_row_timestamp(row["timestamp"])
    
    # Konversi tahun ke 2025, dan tanggal > 7 ke tanggal 5
    new_day = 5 if dt.day > 7 else dt.day
    dt = dt.replace(year=2025, day=new_day)
    
    lat = float(row["latitude"])
    lon = float(row["longitude"])
    tinggi = float(row["tinggi_letusan_m"])

    timestamp = to_hysplit_timestamp(dt)
    met_file = infer_gdas_met_file(dt)

    create_control_file(
        lat_sumber=lat,
        lon_sumber=lon,
        tinggi_letusan_m=tinggi,
        timestamp=timestamp,
        met_file=met_file,
    )

    run_hysplit_once(met_file)
    cdump_txt = convert_cdump_to_txt()
    metrics = analyze_cdump(cdump_txt, source_lat=lat, source_lon=lon, threshold_conc=THRESHOLD_CONC)
    return metrics


def fill_dataset_targets(dataset_path=DATASET_PATH, output_path=OUTPUT_PATH):
    if not dataset_path.exists():
        raise FileNotFoundError(f"dataset tidak ditemukan: {dataset_path}")

    if output_path.exists() and not OVERWRITE_OUTPUT:
        raise FileExistsError(f"output sudah ada: {output_path}")

    df = pd.read_csv(dataset_path)

    missing_inputs = [c for c in REQUIRED_INPUT_COLS if c not in df.columns]
    if missing_inputs:
        raise ValueError(f"kolom input wajib tidak ditemukan: {missing_inputs}")

    for col in TARGET_COLS:
        if col not in df.columns:
            df[col] = np.nan

    pending_mask = df[TARGET_COLS].isna().any(axis=1)
    pending_idx = df.index[pending_mask].tolist()

    if MAX_ROWS is not None:
        pending_idx = pending_idx[: int(MAX_ROWS)]

    print("=" * 70)
    print("BATCH FILL TARGET DATASET")
    print("=" * 70)
    print(f"Dataset         : {dataset_path}")
    print(f"Output          : {output_path}")
    print(f"Runtime simulasi: {BATCH_RUNTIME_JAM} jam")
    print(f"Total baris     : {len(df)}")
    print(f"Baris pending   : {len(pending_idx)}")

    success = 0
    failed = 0

    for n, idx in enumerate(pending_idx, start=1):
        row = df.loc[idx]
        rec_id = row.get("id", idx)
        volcano = row.get("volcano_filter", "unknown")

        print("\n" + "-" * 70)
        print(f"[{n}/{len(pending_idx)}] id={rec_id}, gunung={volcano}")

        try:
            metrics = run_one_row(row)
            for key, value in metrics.items():
                df.at[idx, key] = value

            success += 1
            print(
                f"OK: jarak={metrics['jarak_km']:.2f} km, "
                f"luas={metrics['luas_km2']:.2f} km2, "
                f"sudut={metrics['sudut_deg']:.2f} deg, "
                f"radius={metrics['radius_km']:.2f} km"
            )

            if n % CHECKPOINT_EVERY == 0:
                df.to_csv(output_path, index=False, encoding="utf-8-sig")
                print(f"Checkpoint tersimpan: {output_path}")

        except Exception as e:
            failed += 1
            print(f"Gagal: {e}")

    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 70)
    print("RINGKASAN")
    print("=" * 70)
    print(f"Berhasil: {success}")
    print(f"Gagal   : {failed}")
    print(f"Output  : {output_path}")

    return df


batch_df = fill_dataset_targets()

BATCH FILL TARGET DATASET
Dataset         : d:\Projects\volcanic_ash\simulation\dataset-java-ash.csv
Output          : d:\Projects\volcanic_ash\simulation\dataset-java-ash_hysplit_filled.csv
Runtime simulasi: 1 jam
Total baris     : 1707
Baris pending   : 1707

----------------------------------------------------------------------
[1/1707] id=1, gunung=Merapi
OK: jarak=12.64 km, luas=122.86 km2, sudut=300.15 deg, radius=6.70 km

----------------------------------------------------------------------
[2/1707] id=2, gunung=Merapi
OK: jarak=12.51 km, luas=122.86 km2, sudut=6.10 deg, radius=6.61 km

----------------------------------------------------------------------
[3/1707] id=3, gunung=Merapi
OK: jarak=22.26 km, luas=767.81 km2, sudut=279.52 deg, radius=11.26 km

----------------------------------------------------------------------
[4/1707] id=4, gunung=Merapi
OK: jarak=11.24 km, luas=460.69 km2, sudut=285.84 deg, radius=5.76 km

-------------------------------------------------------